# 01 - Logistic Regression (calibrated linear baseline)

The interpretable **linear baseline** for cancel-by-arrival. Evaluated on the SAME
**decision-time walk-forward** as every model (00 §12 / `src.walkforward`), so it is
directly comparable to the hazard model (08). It uses the **linear feature family**
(skew-damped `_log` twins, scaled, one-hot categoricals), a **cost-optimal**
operating point (not F1), and reports **PR-AUC + Brier** as headline KPIs. The heavy
lifting - fit, calibrate, walk-forward, retrain - lives in `src.training` /
`src.scoring`; this notebook is a thin, readable driver.

Note: this is a horizon-blind per-booking model. It is the baseline the horizon-aware
hazard model must beat on the matched estimand (08 §5).

## 0 - Setup

In [1]:
from __future__ import annotations
import sys, time
from pathlib import Path
_t0 = time.perf_counter()
def _step(m): print(f"  [{time.perf_counter()-_t0:5.2f}s] {m}", flush=True)
_here = Path.cwd().resolve()
while not (_here / "pyproject.toml").exists():
    if _here == _here.parent: raise RuntimeError("project root not found")
    _here = _here.parent
if str(_here) not in sys.path: sys.path.insert(0, str(_here))

import numpy as np, pandas as pd
import plotly.graph_objects as go, plotly.io as pio
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from src import load_clean_reservations, color, data_dir, figures_dir, tables_dir
from src import walkforward as WF
from src.features import load_feature_roster, family_feature_lists
import src.training as T, src.scoring as sc
pio.templates.default = "plotly_white"
BRAND = {n: color(n) for n in ["yellow","blue","green","orange","pink","purple","red"]}
FIG_DIR = figures_dir()/"01_logreg"; FIG_DIR.mkdir(parents=True, exist_ok=True)
TBL_DIR = tables_dir()/"01_logreg"; TBL_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42; pd.set_option("display.max_columns", 80)
_step("setup done (plotly).")

  [ 1.47s] setup done (plotly).


## 1 - Data + linear feature family

Single source of truth: the roster from 00 §11. The **linear** family swaps each
skewed numeric for its `_log` twin (trees would use the raw ones). Company /
nationality / language / check-in fields are already excluded as leakage.

In [2]:
clean = load_clean_reservations()
roster = load_feature_roster()
NUM, CAT = family_feature_lists(roster, "linear")
print(f"linear-family features: {len(NUM)} numeric + {len(CAT)} categorical")
print("  numeric    :", NUM)
print("  categorical:", CAT)
print("  target     : status (1 = cancel at/before arrival)")

linear-family features: 14 numeric + 7 categorical
  numeric    : ['adults_n', 'arrival_dow', 'arrival_month', 'diff_gross_cancellation_fee_log', 'gross_per_night_log', 'has_children', 'has_corporate_code', 'has_group', 'has_promo', 'is_weekend_arrival', 'lead_time_days_log', 'log_gross_amount', 'los_nights_log', 'ratePlan_isSubjectToCityTax']
  categorical: ['cancellationFee_name', 'channelCode', 'guaranteeType', 'property_name', 'ratePlan_category', 'stay_bucket', 'unitGroup_name']
  target     : status (1 = cancel at/before arrival)


## 2 - Decision-time walk-forward evaluation

`src.training.walk_forward_eval` fits the frozen-hp CALIBRATED pipeline on each
fold's train and scores its decision-time test. Honest, leak-free, per-fold +
aggregate AUC / AP / Brier / cost@analytic-threshold.

In [3]:
_step("decision-time walk-forward (fit+calibrate per fold; heavy)...")
wfres = T.walk_forward_eval("logreg", n_folds=12)
pf = pd.DataFrame(wfres["per_fold"]); display(pf.round(4))
print("aggregate (mean +/- std):",
      {k: f"{v['mean']:.4f}+/-{v['std']:.4f}" for k, v in wfres["aggregate"].items()})
fig = go.Figure()
fig.add_scatter(x=pf["origin"], y=pf["auc"], mode="lines+markers", name="AUC", line=dict(color=BRAND["blue"]))
fig.add_scatter(x=pf["origin"], y=pf["ap"],  mode="lines+markers", name="AP (PR-AUC)", line=dict(color=BRAND["orange"]))
fig.update_layout(title="LogReg decision-time walk-forward: per-fold AUC / AP",
                  xaxis_title="fold origin", yaxis_title="score")
fig.show()

  [ 1.62s] decision-time walk-forward (fit+calibrate per fold; heavy)...


/Users/ruby.grambauer/Documents/DEV/OverbookingAnalyse/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/ruby.grambauer/Documents/DEV/OverbookingAnalyse/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/ruby.grambauer/Documents/DEV/OverbookingAnalyse/.venv/lib/python3.12/site-packages/sklearn/linear_model/_

,fold,origin,n_train,n_test,auc,ap,brier,cost
0,0,2026-01-11,142797,2809,0.7255,0.2937,0.1249,34160.0
1,1,2026-01-25,145544,2614,0.7220,0.2901,0.1174,28480.0
2,2,2026-02-08,148566,2768,0.6630,0.2752,0.1317,34640.0
3,3,2026-02-22,151254,2925,0.7065,0.2085,0.1104,27600.0
4,4,2026-03-08,154293,2598,0.7117,0.2514,0.1101,25120.0
5,5,2026-03-22,157347,2372,0.7251,0.2231,0.1131,22960.0
6,6,2026-04-05,159862,3026,0.6892,0.2300,0.1285,34480.0
7,7,2026-04-19,162819,2953,0.6963,0.2264,0.1274,31600.0
8,8,2026-05-03,166002,3146,0.7231,0.2356,0.1184,32560.0
9,9,2026-05-17,169366,3394,0.7497,0.2573,0.1071,30960.0


aggregate (mean +/- std): {'auc': '0.7194+/-0.0293', 'ap': '0.2504+/-0.0282', 'brier': '0.1160+/-0.0119', 'cost': '29120.0000+/-6997.8074'}


## 3 - Pooled out-of-time predictions + calibration

Pool the per-fold OOS predictions into ONE large decision-aligned sample, then
report pooled KPIs and the **reliability curve** (the probabilities feed the
overbooking decision, so calibration matters as much as ranking).

In [4]:
_step("pooled out-of-time predictions...")
oos = T.walk_forward_predict("logreg", n_folds=12)
yv, pv = oos["y_true"].to_numpy(), oos["y_prob"].to_numpy()
print(f"pooled OOS: n={len(oos):,}  base-rate={yv.mean():.3f}  | "
      f"AUC={roc_auc_score(yv,pv):.4f}  AP={average_precision_score(yv,pv):.4f}  "
      f"Brier={brier_score_loss(yv,pv):.4f}")
from sklearn.calibration import calibration_curve
frac_pos, mean_pred = calibration_curve(yv, pv, n_bins=10, strategy="quantile")
fig = go.Figure()
fig.add_scatter(x=mean_pred, y=frac_pos, mode="lines+markers", name="LogReg", line=dict(color=BRAND["blue"]))
fig.add_scatter(x=[0, float(pv.max())], y=[0, float(pv.max())], mode="lines",
                name="perfect", line=dict(color="grey", dash="dash"))
fig.update_layout(title="Calibration (reliability) on pooled OOS",
                  xaxis_title="mean predicted probability", yaxis_title="observed frequency")
fig.show()

  [7261.75s] pooled out-of-time predictions...


/Users/ruby.grambauer/Documents/DEV/OverbookingAnalyse/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/ruby.grambauer/Documents/DEV/OverbookingAnalyse/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/ruby.grambauer/Documents/DEV/OverbookingAnalyse/.venv/lib/python3.12/site-packages/sklearn/linear_model/_

pooled OOS: n=33,645  base-rate=0.130  | AUC=0.7157  AP=0.2422  Brier=0.1170


## 4 - Cost-optimal operating point (not F1)

Walking a guest is far costlier than an empty room, so the decision threshold is
**cost-minimising**, not F1-balanced (`src.scoring`). We tune it on the pooled OOS.

In [5]:
thr = sc.cost_threshold_from_scores(yv, pv)
op = sc.cost_at_threshold(yv, pv, thr)
print(f"cost-optimal threshold = {thr:.3f}  (walk={sc.COST_WALK:.0f} / empty={sc.COST_EMPTY:.0f})")
print(f"  precision={op['precision']:.3f}  recall={op['recall']:.3f}  total_cost={op['total_cost']:,.0f}")
grid = np.linspace(0.05, 0.95, 60)
costs = [sc.cost_at_threshold(yv, pv, t)["total_cost"] for t in grid]
fig = go.Figure(go.Scatter(x=grid, y=costs, mode="lines", line=dict(color=BRAND["orange"])))
fig.add_vline(x=thr, line=dict(color=BRAND["green"], dash="dash"),
              annotation_text=f"cost-opt {thr:.2f}")
fig.update_layout(title="Total cost vs decision threshold (minimum = operating point)",
                  xaxis_title="threshold", yaxis_title="total cost")
fig.show()

cost-optimal threshold = 0.749  (walk=300 / empty=80)
  precision=0.000  recall=0.000  total_cost=349,440


## 5 - Explainability (XAI)

A linear model is its own explanation: coefficients are log-odds contributions.
We fit an UNCALIBRATED linear pipeline on all resolved data for the coefficients,
and add permutation importance (AP drop) as a model-agnostic cross-check.

In [6]:
df = WF.add_outcome_known_date(clean)
known = pd.to_datetime(df["outcome_known_date"], utc=True, errors="coerce")
resolved = known <= known.max()
X, y = df[NUM + CAT], pd.to_numeric(df["status"], errors="coerce").astype(int)

lin = T.build_pipeline("logreg", {"C": 1.0, "l1_ratio": 0.5}, NUM, CAT, calibrate=False, seed=RANDOM_STATE)
lin.fit(X[resolved], y[resolved].to_numpy())
names = lin.named_steps["prep"].get_feature_names_out()
coef = lin.named_steps["clf"].coef_[0]
cdf = pd.DataFrame({"feature": names, "coef": coef}).sort_values("coef")
top = pd.concat([cdf.head(10), cdf.tail(10)])
fig = go.Figure(go.Bar(x=top["coef"], y=top["feature"], orientation="h",
                       marker_color=[BRAND["red"] if c < 0 else BRAND["green"] for c in top["coef"]]))
fig.update_layout(title="LogReg coefficients (log-odds; + raises cancel risk)", xaxis_title="coefficient")
fig.show()

from sklearn.inspection import permutation_importance
samp = X[resolved].sample(n=min(5000, int(resolved.sum())), random_state=RANDOM_STATE)
pi = permutation_importance(lin, samp, y[resolved].loc[samp.index],
                            scoring="average_precision", n_repeats=5, random_state=RANDOM_STATE)
pidf = (pd.DataFrame({"feature": NUM + CAT, "importance": pi.importances_mean})
          .sort_values("importance", ascending=False).head(15))
fig = go.Figure(go.Bar(x=pidf["importance"][::-1], y=pidf["feature"][::-1],
                       orientation="h", marker_color=BRAND["blue"]))
fig.update_layout(title="Permutation importance (mean AP decrease)", xaxis_title="AP drop")
fig.show()

/Users/ruby.grambauer/Documents/DEV/OverbookingAnalyse/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


## 6 - Deployment model + persist

`src.training.retrain(mode="retune")` re-searches C / l1_ratio (TimeSeriesSplit, AP),
fits the calibrated pipeline on ALL resolved data, and persists the joblib + card
the app's scoring path loads. It reports the same decision-time walk-forward metrics.

In [7]:
_step("deployment fit (retune) + persist...")
deploy = T.retrain("logreg", mode="retune")
print("persisted ->", deploy["persisted"]["joblib"])
print("trained on", deploy["n_train_deploy"], "resolved rows | chosen hp:", deploy["hyperparams"])
print("walk-forward aggregate:",
      {k: round(v["mean"], 4) for k, v in deploy["walk_forward"]["aggregate"].items()})

  [11390.00s] deployment fit (retune) + persist...
[retrain:logreg] roster changed vs deployed model - added=['adults_n', 'arrival_dow', 'arrival_month', 'cancellationFee_name', 'channelCode', 'diff_gross_cancellation_fee', 'diff_gross_cancellation_fee_log', 'gross_per_night', 'gross_per_night_log', 'guaranteeType', 'has_children', 'has_corporate_code', 'has_group', 'has_promo', 'is_weekend_arrival', 'lead_time_days', 'lead_time_days_log', 'log_gross_amount', 'los_nights', 'los_nights_log', 'property_name', 'ratePlan_category', 'ratePlan_isSubjectToCityTax', 'stay_bucket', 'unitGroup_name'] removed=[]


/Users/ruby.grambauer/Documents/DEV/OverbookingAnalyse/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/ruby.grambauer/Documents/DEV/OverbookingAnalyse/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/ruby.grambauer/Documents/DEV/OverbookingAnalyse/.venv/lib/python3.12/site-packages/sklearn/linear_model/_

persisted -> /Users/ruby.grambauer/Documents/DEV/OverbookingAnalyse/Data/01_logreg_model.joblib
trained on 179576 resolved rows | chosen hp: {'C': np.float64(0.15577217702693022), 'l1_ratio': np.float64(0.7851759613930136)}
walk-forward aggregate: {'auc': 0.7297, 'ap': 0.2426, 'brier': 0.1127, 'cost': 29413.3333}


## 7 - Verdict

The calibrated LogReg is the transparent, horizon-blind baseline: a linear log-odds
model whose coefficients say exactly why a booking is flagged, evaluated honestly on
the decision-time walk-forward and calibrated for the cost-based overbooking decision.
Its role is the yardstick the horizon-aware hazard model (08 §5) must beat on the
matched estimand; where it loses to the hazard near arrival is precisely the
time-varying risk a linear per-booking model cannot represent.